In [ ]:
from Run_Pipeline.Agent_Tool import AgentTools
from Run_Pipeline.Document_Loader import DocumentLoader
from Run_Pipeline.Document_Splitter import DocumentSplitter
from Run_Pipeline.Embedding_DB import EmbeddingDB
from Run_Pipeline.Env_Loader import (EnvLoader)
from Run_Pipeline.LLM_Factory import LLMFactory
from Run_Pipeline.Retriever_Builder import RetrieverBuilder
from pathlib import Path
import os

EnvLoader.load_local(dotenv_path="../.env")

BASE_DIR = Path(os.getcwd()).parent

kind = "json"
file_path = BASE_DIR / os.getenv("DATA_PATH")
chunk_size = 512
overlap_size = 50
device = "mps"
persist_dir = BASE_DIR / os.getenv("DATA_PATH") / f"{kind}_{chunk_size}"
retriever_mode = 3
k = 3
engine_num = 1
backend_num = 1



In [ ]:
def build_chain():
    """
    RAG 파이프라인을 초기화해 LangSmith가 매 예제마다 새로 씁니다.
    """
    # --- ☑️ 당신이 앞서 작성한 코드 재사용 ---
    loader = DocumentLoader(file_path, kind)
    docs_post, docs_company = loader._load_json_files()

    splitter = DocumentSplitter(chunk_size=512, overlap=50)
    chunks = splitter.split(docs_post + docs_company, cache_dir=file_path)

    embed_db = EmbeddingDB(
        model_name="nlpai-lab/KURE-v1",
        device="mps",
        persist_dir=persist_dir
    )
    db = embed_db.get_or_create_db(chunks)

    retriever = RetrieverBuilder(
        db=db,
        full_docs=docs_post + docs_company,
        chunks=chunks,
        k=3,
    ).build(mode=3)

    llm = LLMFactory.load_llm(engine=1, backend=1, device="mps")

    from Run_Pipeline.Chain_Factory import ChainFactory
    return ChainFactory.build_chain(retriever=retriever, llm=llm)


In [ ]:
import pandas as pd

df = pd.read_excel('../Data_Files/대통합.xlsx')
my_questions = df['Querry'].to_list()

In [ ]:
# rag_to_langsmith.py
"""질문 리스트를 RAG 체인에 통과시켜 (질문, 답변, 맥락)을 LangSmith 데이터셋에 업로드하는 스크립트.

사용법 예시:

    from your_build_module import build_chain  # 기존 build_chain 가져오기
    from rag_to_langsmith import upload_qac_dataset

    questions = [
        "회사의 핵심 사업은 무엇인가요?",
        "재택근무 제도는 있나요?",
        # ...
    ]

    upload_qac_dataset(build_chain, questions, dataset_name="rag_qac_dataset")
"""

from __future__ import annotations

import json
from typing import Callable, Dict, List, Any

from langsmith import Client
from langchain.schema import Document

###############################################################################
# LangSmith 업로드 유틸
###############################################################################

def _upload_to_langsmith(records: List[Dict[str, str]], dataset_name: str) -> str:
    """records = [{question, answer, context}, ...] 형태를 LangSmith에 업로드."""
    client = Client()

    # 기존 동일 이름 데이터셋이 있으면 삭제
    for ds in client.list_datasets():
        if ds.name == dataset_name:
            client.delete_dataset(dataset_id=ds.id)
            break

    dataset = client.create_dataset(dataset_name=dataset_name)

    for idx, r in enumerate(records, 1):
        client.create_example(
            dataset_id=dataset.id,
            inputs={"question": r["question"]},
            outputs={
                "answer": r["answer"],
                "context": r["context"],
            },
            metadata={"row": idx},
        )

    print(f"[LangSmith] '{dataset_name}' 업로드 완료! (총 {len(records)}개)")
    return dataset_name

###############################################################################
# RAG 체인 실행 결과에서 answer / context 추출 유틸
###############################################################################


def _extract_answer(result: Any) -> str:
    """체인 invoke 결과에서 답변 문자열 추출."""
    if isinstance(result, str):
        return result
    if isinstance(result, dict):
        for key in ("answer", "result", "output"):
            if key in result:
                return str(result[key])
    # LangChain Runnable 시나리오: LLMResult list 등
    return str(result)


def _extract_context(result: Any) -> str:
    """체인 invoke 결과에서 맥락(source_documents) 텍스트 추출."""
    # 1) attr 로 존재할 때
    if hasattr(result, "source_documents"):
        docs = getattr(result, "source_documents")
        return "\n\n".join(doc.page_content for doc in docs)

    # 2) dict 구조
    if isinstance(result, dict):
        if "source_documents" in result:
            return "\n\n".join(doc.page_content for doc in result["source_documents"])
        if "context" in result:
            return str(result["context"])
        if "retrieved_docs" in result:
            return "\n\n".join(str(d) for d in result["retrieved_docs"])

    return "(맥락 추출 불가)"

###############################################################################
# 메인 함수
###############################################################################


def upload_qac_dataset(
    build_chain_fn: Callable[[], Any],
    questions: List[str],
    dataset_name: str = "rag_qac_dataset",
    show_progress: bool = True,
) -> str:
    """build_chain 함수와 질문 리스트를 받아 LangSmith 데이터셋 생성.

    Args:
        build_chain_fn: 빌드된 Chain 을 반환하는 콜러블 (ex. build_chain)
        questions: 문자열 질문 리스트
        dataset_name: LangSmith 상에 생성할 데이터셋 이름
        show_progress: True면 진행 상황 print
    Returns:
        dataset_name (str)
    """

    records: List[Dict[str, str]] = []
    for idx, q in enumerate(questions, 1):
        if show_progress:
            print(f"[{idx}/{len(questions)}] Q: {q[:30]}…", end=" ")
        try:
            result = chain.invoke(f"question : {q}")
            answer = _extract_answer(result)
            context = _extract_context(result)
        except Exception as e:
            answer = f"오류 발생: {e}"
            context = ""
        records.append({"question": q, "answer": answer, "context": context})
        if show_progress:
            print("✅")

    return _upload_to_langsmith(records, dataset_name)

###############################################################################
# CLI 사용용 샘플
###############################################################################

if __name__ == "__main__":
    # --- 사용자 영역 수정 -----------------------------------------------

    # --------------------------------------------------------------------

    upload_qac_dataset(build_chain, my_questions, dataset_name="rag_qac_dataset_Retriever_3")


# LangSmith 데이터셋에서 가져오기 이름만 바꾸면 됨

In [ ]:
# 1. LangSmith 클라이언트 준비
from langsmith import Client
import pandas as pd          # 표로 보고 싶다면

client = Client()

# 2. 데이터셋 객체 찾기  ───────────────────────────────
DATASET_NAME = "rag_qac_dataset"        # ↔︎ 실제 이름
ds = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)

# 3. 예제(샘플) 목록 가져오기
examples = client.list_examples(dataset_id=ds.id)

# 4. 원하는 컬럼만 추려서 확인
records = []
for ex in examples:
    q = ex.inputs.get("question", "")
    a = ex.outputs.get("answer", "")
    ctx = ex.outputs.get("context", "")          # 또는 ex.outputs["context"]
    records.append({"question": q, "context": ctx, "answer": a})

# 5-a. 터미널에서 빠르게 보기
for r in records[:3]:
    print("Q:", r["question"])
    print("A:", r["answer"][:80], "…")
    print("CTX:", r["context"][:80], "…")
    print("-"*40)

# 5-b. 판다스 데이터프레임으로
df = pd.DataFrame(records)
df



# LLM 평가기

In [21]:
import pandas as pd
import json
import time
from typing import Dict, List, Optional
import google.generativeai as genai
import openai
from langsmith import Client

class RAGEvaluator:
    def __init__(self, llm_provider: str = "gemini", api_key: str = None):
        """
        RAG 평가기 초기화

        Args:
            llm_provider: "gemini" 또는 "perplexity"
            api_key: API 키
        """
        self.llm_provider = llm_provider.lower()

        if self.llm_provider == "gemini":
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel('gemini-2.0-flash')
        elif self.llm_provider == "perplexity":
            self.client = openai.OpenAI(
                api_key=api_key,
                base_url="https://api.perplexity.ai"
            )
        else:
            raise ValueError("llm_provider는 'gemini' 또는 'perplexity'여야 합니다.")

    def create_evaluation_prompt(self, question: str, context: str, answer: str, metric_type: str) -> str:
        """평가 메트릭별 프롬프트 생성"""

        base_instruction = f"""
다음 RAG 시스템의 성능을 평가해주세요.

**질문 (Question):**
{question}

**검색된 맥락 (Context):**
{context}

**생성된 답변 (Answer):**
{answer}

---
"""

        if metric_type == "faithfulness":
            specific_instruction = """
**평가 기준: 신뢰성 (Faithfulness)**
생성된 답변이 검색된 맥락 데이터에 얼마나 사실적으로 부합하는지 평가하세요.
- 10점: 답변의 모든 내용이 맥락에서 직접 뒷받침됨
- 8-9점: 답변의 대부분이 맥락에 부합하며, 약간의 추론이 있음
- 6-7점: 답변의 일부가 맥락에 부합하지만 일부 내용이 불분명
- 4-5점: 답변의 일부만 맥락에 부합하며 상당한 추측이 포함됨
- 2-3점: 답변이 맥락과 크게 다르거나 모순됨
- 0-1점: 답변이 맥락과 전혀 부합하지 않음
"""

        elif metric_type == "answer_relevance":
            specific_instruction = """
**평가 기준: 답변 관련성 (Answer Relevance)**
생성된 답변이 질문과 얼마나 관련성이 있고 적절한지 평가하세요.
- 10점: 질문에 완벽하게 답변하며 정확하고 완전함
- 8-9점: 질문에 잘 답변하지만 약간의 부족한 부분이 있음
- 6-7점: 질문에 부분적으로 답변하지만 핵심을 놓침
- 4-5점: 질문과 관련은 있지만 부적절하거나 불완전함
- 2-3점: 질문과 약간 관련이 있지만 대부분 부적절함
- 0-1점: 질문과 전혀 관련이 없음
"""

        elif metric_type == "context_relevance":
            specific_instruction = """
**평가 기준: 맥락 관련성 (Context Relevance)**
검색된 맥락 데이터가 질문과 얼마나 관련이 있고 유용한지 평가하세요.
- 10점: 맥락이 질문에 완벽하게 관련되고 답변에 필요한 모든 정보 포함
- 8-9점: 맥락이 질문에 매우 관련되며 대부분의 필요한 정보 포함
- 6-7점: 맥락이 질문에 관련되지만 일부 정보가 부족하거나 불필요함
- 4-5점: 맥락이 질문에 부분적으로만 관련됨
- 2-3점: 맥락이 질문과 약간만 관련됨
- 0-1점: 맥락이 질문과 전혀 관련이 없음
"""

        output_format = """
**출력 형식:**
다음 JSON 형식으로만 응답해주세요:
{
    "score": [0-10 점수],
    "reasoning": "평가 근거에 대한 간단한 설명"
}
"""

        return base_instruction + specific_instruction + output_format

    def get_llm_response(self, prompt: str) -> str:
        """LLM에서 응답 받기"""
        try:
            if self.llm_provider == "gemini":
                response = self.model.generate_content(prompt)
                return response.text

            elif self.llm_provider == "perplexity":
                response = self.client.chat.completions.create(
                    model="llama-3.1-sonar-large-128k-online",  # 또는 다른 모델
                    messages=[
                        {"role": "system", "content": "당신은 RAG 시스템 평가 전문가입니다. 정확하고 객관적인 평가를 수행하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1
                )
                return response.choices[0].message.content

        except Exception as e:
            print(f"LLM 응답 오류: {e}")
            return None

    def parse_llm_response(self, response: str) -> Dict:
        """LLM 응답에서 점수와 근거 추출"""
        try:
            # JSON 부분만 추출
            start_idx = response.find('{')
            end_idx = response.rfind('}') + 1
            json_str = response[start_idx:end_idx]

            result = json.loads(json_str)

            # 점수 유효성 검사
            score = result.get('score', 0)
            if not isinstance(score, (int, float)) or score < 0 or score > 10:
                score = 0

            return {
                'score': float(score),
                'reasoning': result.get('reasoning', 'No reasoning provided')
            }

        except Exception as e:
            print(f"응답 파싱 오류: {e}")
            print(f"원본 응답: {response}")
            return {'score': 0.0, 'reasoning': 'Parsing failed'}

    def evaluate_single_example(self, question: str, context: str, answer: str) -> Dict:
        """단일 예제에 대한 전체 평가"""
        metrics = ['faithfulness', 'answer_relevance', 'context_relevance']
        results = {}

        for metric in metrics:
            print(f"  평가 중: {metric}")

            prompt = self.create_evaluation_prompt(question, context, answer, metric)
            response = self.get_llm_response(prompt)

            if response:
                parsed = self.parse_llm_response(response)
                results[metric] = parsed
            else:
                results[metric] = {'score': 0.0, 'reasoning': 'LLM response failed'}

            # API 호출 간격 (rate limiting 방지)
            time.sleep(1)

        return results

    def evaluate_dataset(self, records: List[Dict]) -> pd.DataFrame:
        """전체 데이터셋 평가"""
        all_results = []

        for i, record in enumerate(records):
            print(f"\n평가 진행 중... ({i+1}/{len(records)})")

            question = record['question']
            context = record['context']
            answer = record['answer']

            # 각 메트릭별 평가
            evaluation_results = self.evaluate_single_example(question, context, answer)

            # 결과 저장
            result_row = {
                'question': question,
                'context': context,
                'answer': answer,
                'faithfulness_score': evaluation_results['faithfulness']['score'],
                'faithfulness_reasoning': evaluation_results['faithfulness']['reasoning'],
                'answer_relevance_score': evaluation_results['answer_relevance']['score'],
                'answer_relevance_reasoning': evaluation_results['answer_relevance']['reasoning'],
                'context_relevance_score': evaluation_results['context_relevance']['score'],
                'context_relevance_reasoning': evaluation_results['context_relevance']['reasoning'],
                'average_score': (
                    evaluation_results['faithfulness']['score'] +
                    evaluation_results['answer_relevance']['score'] +
                    evaluation_results['context_relevance']['score']
                ) / 3
            }

            all_results.append(result_row)

        return pd.DataFrame(all_results)

# 사용 예시


In [24]:
def main():
    # 1. LangSmith에서 데이터셋 가져오기 (기존 코드)
    client = Client()
    DATASET_NAME = "rag_qac_dataset"
    ds = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)
    examples = client.list_examples(dataset_id=ds.id)

    records = []
    for ex in examples:
        q = ex.inputs.get("question", "")
        a = ex.outputs.get("answer", "")
        ctx = ex.outputs.get("context", "")
        records.append({"question": q, "context": ctx, "answer": a})

    # 2. 평가기 초기화
    API_KEY = "."  # 실제 API 키로 교체

    # Gemini 사용하는 경우
    evaluator = RAGEvaluator(llm_provider="gemini", api_key=API_KEY)

    # Perplexity 사용하는 경우
    #evaluator = RAGEvaluator(llm_provider="perplexity", api_key=API_KEY)

    # 3. 평가 실행 (전체 데이터셋 또는 샘플)
    #sample_records = records[:5]  # 테스트용으로 5개만
    sample_records = records  # 전체 데이터셋 평가

    print("RAG 시스템 평가를 시작합니다...")
    results_df = evaluator.evaluate_dataset(sample_records)

    # 4. 결과 저장 및 요약
    results_df.to_csv('rag_evaluation_results.csv', index=False, encoding='utf-8-sig')

    # 요약 통계
    print("\n=== 평가 결과 요약 ===")
    print(f"총 평가 항목 수: {len(results_df)}")
    print(f"신뢰성 평균 점수: {results_df['faithfulness_score'].mean():.2f}")
    print(f"답변 관련성 평균 점수: {results_df['answer_relevance_score'].mean():.2f}")
    print(f"맥락 관련성 평균 점수: {results_df['context_relevance_score'].mean():.2f}")
    print(f"전체 평균 점수: {results_df['average_score'].mean():.2f}")

    # 상세 결과 출력
    print("\n=== 상세 결과 ===")
    print(results_df[['faithfulness_score', 'answer_relevance_score', 'context_relevance_score', 'average_score']].head())

    return results_df

if __name__ == "__main__":
esults = main()

RAG 시스템 평가를 시작합니다...

평가 진행 중... (1/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (2/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (3/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (4/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (5/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (6/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (7/32)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance
LLM 응답 오류: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerM

In [27]:
results.head(30)

,question,context,answer,faithfulness_score,faithfulness_reasoning,answer_relevance_score,answer_relevance_reasoning,context_relevance_score,context_relevance_reasoning,average_score
0,영상화/멀티미디어 제작사 관련 공고 보여줘,제목: 제작 PD 광고 및 홍보 영상 콘텐츠 제작\n포지션 상세 설명: 알파브라더...,현재 영상화/멀티미디어 제작사와 관련된 공고는 다음과 같습니다:\n\n### 1. ...,10.0,"답변의 모든 내용 (알파브라더스, 어댑트 공고의 상세 내용, 근무지, 신입 지원 가...",10.0,"질문에서 요구한 '영상화/멀티미디어 제작사 관련 공고'를 정확하게 찾아 제시했으며,...",10.0,질문에서 요구하는 영상화/멀티미디어 제작사 관련 공고를 정확하게 찾아내어 요약했습니...,10.000000
1,서울에 있는 GMO/식품/바이오 관련 회사 공고 정리해줘,회사명: 강남엄마\n회사 소개: 1 강남엄마의 사업 목적은 교육 시장의 정보 비대칭...,서울에 있는 GMO/식품/바이오 관련 회사의 공고를 정리해 드리겠습니다. 아래는 해...,9.0,답변은 검색된 맥락에 매우 충실합니다. 바이오일레븐과 지놈오피니언의 정보는 맥락에서...,9.0,질문에서 요구한 서울에 있는 GMO/식품/바이오 관련 회사 정보를 잘 정리하여 제공...,9.0,맥락은 질문에 매우 관련성이 높다. 질문에서 요구하는 '서울에 있는 GMO/식품/바...,9.000000
2,UX디자이너 채용 공고 좀 보여줘,제목: UI UX 디자이너\n포지션 상세 설명: 회사 소개스트리밍이 주도하는 현대...,다음은 UX 디자이너 채용 공고의 예시입니다.\n\n### 1. 뮤즈라이브 (Muz...,9.0,생성된 답변은 검색된 맥락 데이터에 기반하여 UX 디자이너 채용 공고를 요약하고 있...,10.0,"질문에서 요구한 UX 디자이너 채용 공고를 잘 요약하여 보여주고 있으며, 각 공고의...",9.0,질문 'UX디자이너 채용 공고 좀 보여줘'에 대해 제공된 맥락은 세 개의 회사(뮤즈...,9.333333
3,디지털 마케팅 인턴 공고를 보여줘,제목: 인턴 일본 마케팅 채용형\n포지션 상세 설명: 인턴공고는 정규직으로 채용될 ...,디지털 마케팅 인턴 공고는 다음과 같습니다:\n\n### 인턴 일본 마케팅 채용형\...,9.0,"답변은 검색된 맥락에서 대부분 정보를 가져왔으며, 약간의 정보는 요약되었습니다. 예...",9.0,"질문에서 요구한 '디지털 마케팅 인턴 공고'를 맥락에서 정확히 찾아냈고, 관련 정보...",9.0,"검색된 맥락에서 ""디지털 마케팅 인턴 공고""에 대한 질문에 가장 적합한 정보를 정확...",9.000000
4,서울 금천구에 있는 PR회사 공고를 보여줘,제목: PR 크리에이터\n포지션 상세 설명: To be the Global No1 ...,서울 금천구에 있는 PR 회사의 공고는 다음과 같습니다:\n\n**회사명:** 인터...,10.0,요청사항에 따라 금천구 PR 회사 공고인 인터엑스 정보를 정확하게 요약하여 제공하고...,10.0,"질문에서 요구한 ""서울 금천구에 있는 PR회사 공고""에 대해 정확하고 완전하게 답변...",10.0,"맥락이 질문에 완벽하게 관련되며, 서울 금천구에 있는 PR회사(인터엑스)의 공고를 ...",10.000000
5,재택 근무를 할 수 있는 회사 공고 알려줘,제목: 디지털 캠페인 기획자 2년 이상\n포지션 상세 설명: 세계적인 Stagwel...,재택 근무를 지원하는 회사 공고는 다음과 같습니다:\n\n1. **앨리슨하이퍼앰**...,10.0,"답변이 맥락에서 제공된 정보를 정확하게 요약하고 있으며, 재택 근무 관련 정보를 충...",10.0,질문에서 요구하는 재택근무를 지원하는 회사 정보를 정확하게 추출하여 제공하고 있습니...,10.0,질문에서 요구하는 '재택 근무를 할 수 있는 회사 공고'에 대한 정보를 맥락에서 정...,10.000000
6,고졸로 지원할 수 있는 대기업 공고를 알려줘,제목: Midlevel Full Stack Developer 전문연구요원 가능\n포...,"고졸로 지원할 수 있는 대기업 공고는 일반적으로 특정 직무나 산업에 따라 다르며, ...",2.0,맥락으로 주어진 정보는 고졸 채용과는 전혀 관련이 없는 Midlevel Full S...,4.0,"검색된 맥락에는 고졸 채용 공고가 없는데, 답변은 고졸 채용이 가능하다는 일반적인 ...",0.0,LLM response failed,2.000000
7,Growth Marketer직무 관련 공고 보여줘,제목: 마케팅 세일즈 매니저 신입\n포지션 상세 설명: Making everythi...,"현재 제공된 정보에는 ""Growth Marketer"" 직무에 대한 구체적인 공고가 ...",8.0,"질문이 특정 직무 공고를 찾는 것이었으나, 제공된 맥락에 정확히 일치하는 공고는 없...",0.0,LLM response failed,0.0,LLM response failed,2.666667
8,프레인글로벌에 입사하고 싶은데 다른 나라에도 지사가 있는지 알고 싶어,제목: 언론미디어 PR 과차장급\n포지션 상세 설명: 국내 1위 PR 기업 프레인...,"프레인글로벌은 국내 1위 PR 기업으로, 해외 PR AWARDS에서 대상을 수상한 ...",0.0,LLM response failed,0.0,LLM response failed,0.0,LLM response failed,0.000000
9,KB국민은행에 어떤 직군이 있는지 알려줘,제목: 재무 매니저 Financial Manager\n포지션 상세 설명: The P...,"KB국민은행은 다양한 직군을 보유하고 있으며, 일반적으로 다음과 같은 직군이 있습니...",0.0,LLM response failed,0.0,LLM response failed,0.0,LLM response failed,0.000000


In [23]:
results.to_json('rag_evaluation_results.json', orient='records', force_ascii=False, indent=4)

,question,context,answer,faithfulness_score,faithfulness_reasoning,answer_relevance_score,answer_relevance_reasoning,context_relevance_score,context_relevance_reasoning,average_score
0,영상화/멀티미디어 제작사 관련 공고 보여줘,제목: 제작 PD 광고 및 홍보 영상 콘텐츠 제작\n포지션 상세 설명: 알파브라더...,현재 영상화/멀티미디어 제작사와 관련된 공고는 다음과 같습니다:\n\n### 1. ...,10.0,"답변의 모든 내용 (알파브라더스, 어댑트 공고 요약)이 주어진 맥락에서 정확하게 뒷...",10.0,질문에서 요구한 영상화/멀티미디어 제작사 관련 공고를 정확하게 추출하여 필요한 정보...,10.0,"검색된 맥락 데이터는 질문에 완벽하게 관련되어 있으며, 영상화/멀티미디어 제작사 관...",10.0
1,서울에 있는 GMO/식품/바이오 관련 회사 공고 정리해줘,회사명: 강남엄마\n회사 소개: 1 강남엄마의 사업 목적은 교육 시장의 정보 비대칭...,서울에 있는 GMO/식품/바이오 관련 회사의 공고를 정리해 드리겠습니다. 아래는 해...,9.0,답변은 맥락에 기반하여 정확하게 정보를 제공하고 있습니다. 바이오일레븐과 지놈오피니...,9.0,"질문에 대한 답변으로, 서울에 위치한 바이오 관련 회사들의 정보를 잘 정리하여 제공...",9.0,"맥락이 질문에 매우 관련이 높고, 바이오 관련 회사 정보를 제공합니다. '강남엄마'...",9.0
2,UX디자이너 채용 공고 좀 보여줘,제목: UI UX 디자이너\n포지션 상세 설명: 회사 소개스트리밍이 주도하는 현대...,다음은 UX 디자이너 채용 공고의 예시입니다.\n\n### 1. 뮤즈라이브 (Muz...,9.0,답변은 검색된 맥락을 바탕으로 UX 디자이너 채용 공고를 요약하고 있어 신뢰성이 높...,9.0,질문에 매우 잘 답변했습니다. 검색된 맥락을 바탕으로 UX 디자이너 채용 공고를 정...,9.0,검색된 맥락은 질문에 매우 관련이 높습니다. 질문에서 UX 디자이너 채용 공고를 요...,9.0
3,디지털 마케팅 인턴 공고를 보여줘,제목: 인턴 일본 마케팅 채용형\n포지션 상세 설명: 인턴공고는 정규직으로 채용될 ...,디지털 마케팅 인턴 공고는 다음과 같습니다:\n\n### 인턴 일본 마케팅 채용형\...,9.0,답변은 맥락에 부합하며 필요한 정보를 잘 요약했습니다. 다만 '다른 디지털 마케팅 ...,9.0,"질문에 완벽하게 부합하는 답변을 제공했습니다. 디지털 마케팅 인턴 공고를 요청했고,...",9.0,"질문이 '디지털 마케팅 인턴 공고'를 요청했고, 첫 번째 맥락인 그래비티랩스의 '인...",9.0
4,서울 금천구에 있는 PR회사 공고를 보여줘,제목: PR 크리에이터\n포지션 상세 설명: To be the Global No1 ...,서울 금천구에 있는 PR 회사의 공고는 다음과 같습니다:\n\n**회사명:** 인터...,10.0,"요청된 질문에 대해 검색된 맥락에서 회사명, 직무, 포지션 상세 설명, 주요 업무,...",10.0,질문에서 요구한 금천구 소재 PR 회사 공고(인터엑스)를 정확하게 찾아 상세 내용(...,10.0,검색된 맥락은 질문에 완벽하게 부합합니다. 질문에서 요구하는 '서울 금천구 PR 회...,10.0
